[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BSLJunhyeonJeon/AI_COP/blob/main/session4/notebooks/02_pose_demo.ipynb)

# session4 · 02 · 웹캠 포즈 데모 — 손(21점) 과 전신(17점)

- **이 노트북에서 배우는 것**: 같은 사진에서 **손 랜드마크 21개**(MediaPipe)와 **전신 키포인트 17개**(YOLO11n-pose)를 뽑아 본다. 3회차의 그 YOLO 가 백본은 그대로인 채 **헤드만 바뀌어** '네모' 대신 '관절'을 내놓는 것을 확인한다.
- **입력**: 웹캠 정지 사진 1장 (거부/실패 시 업로드 → 샘플 이미지로 자동 폴백).
- **출력**: `outputs/04_hand.png` (손 21점) · `outputs/04_pose.png` (전신 17점)

---

## ⚠️ 실행 순서 — 이 노트북은 **맨 마지막**에 여세요

이 노트북은 **`mediapipe` 를 설치**합니다. mediapipe 는 numpy·protobuf 등의 버전을 함께 끌고 와서
**런타임의 패키지 구성을 바꿀 수 있습니다**(3회차에서 ultralytics 가 torch 를 덮어썼는지 검사했던 것과 같은 이유).

> **`01_train_lab` → `03_build_html` 을 먼저 끝낸 뒤, 이 노트북을 마지막에 실행하세요.**
> 문제가 생기면 **런타임 재시작 후 이 노트북만 다시 열면** 됩니다. 01/03 의 결과물(`outputs/`)은 그대로 남습니다.

> 이 노트북의 결과 그림(`04_hand.png`·`04_pose.png`)은 **개인 얼굴·손 사진**이라 레포·강의 HTML 에 넣지 않습니다. 라이브에서 화면으로만 봅니다.

In [ ]:
# 셀 1 · 환경 감지 + 프로젝트 루트 확보 (01 과 동일 패턴 — 분기는 이 셀 한 곳)
import os, subprocess

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

SESSION = "session4"
REPO_URL = "https://github.com/BSLJunhyeonJeon/AI_COP"
REPO_DIR = "/content/AI_COP"
SESSION_DIR = REPO_DIR + "/" + SESSION


def acquire_project():
    if os.path.isdir(REPO_DIR):
        print("이미 존재:", REPO_DIR, "(재클론 건너뜀)")
        try:
            r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
            if r.returncode != 0:
                print("  (git pull 실패 — 기존 캐시 버전 사용)")
        except Exception as e:
            print("  (git pull 건너뜀:", e, ")")
    else:
        print("레포 클론:", REPO_URL, "->", REPO_DIR)
        try:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        except Exception as e:
            print("clone 실패(네트워크/권한 확인):", e)
    return SESSION_DIR if os.path.isdir(SESSION_DIR) else None


def find_root_local(marker="requirements.txt"):
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, marker)):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            print("[주의] '" + marker + "' 를 못 찾음. 현재 폴더를 루트로 가정:", start)
            return start
        d = parent


PROJECT_ROOT = acquire_project() if IN_COLAB else find_root_local()
if not (PROJECT_ROOT and os.path.isdir(PROJECT_ROOT)):
    raise RuntimeError(
        "세션 루트를 확보하지 못했습니다. "
        "코랩이면 레포 클론 실패이니 네트워크 확인 후 이 셀(셀 1)을 다시 ▶ 실행하세요. "
        "로컬이면 session4/ 안에서 노트북을 열었는지 확인하세요."
    )
os.chdir(PROJECT_ROOT)
for d in ("data", "weights", "outputs"):
    os.makedirs(d, exist_ok=True)
print("실행 환경   :", "Colab" if IN_COLAB else "Local")
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
# 셀 2 · mediapipe 설치 + 모델 번들 다운로드 + 안전 점검
# mediapipe 는 requirements.txt 에 넣지 않는다(01 의 학습 환경과 분리). 버전도 아직 고정하지 않는다 —
# 코랩에서 실제로 돌려 보고 안전이 확인되면 그때 핀을 확정한다(CONVENTIONS 규칙 3).
import os, sys, subprocess, urllib.request

print("[설치 전 버전]")
BEFORE = {}
for mod in ["numpy", "torch", "ultralytics"]:
    try:
        BEFORE[mod] = getattr(__import__(mod), "__version__", "?")
    except Exception:
        BEFORE[mod] = "(없음)"
    print("  -", mod, ":", BEFORE[mod])

print("\nmediapipe 설치 중... (1~2분)")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mediapipe"], check=False)
print("pip 종료코드:", r.returncode)

# 손 랜드마크 모델 번들 (멱등 — 이미 있으면 건너뜀)
TASK_URL = ("https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
            "hand_landmarker/float16/1/hand_landmarker.task")
TASK_PATH = os.path.join("weights", "hand_landmarker.task")
os.makedirs("weights", exist_ok=True)
if os.path.exists(TASK_PATH) and os.path.getsize(TASK_PATH) > 0:
    print("모델 번들 이미 있음:", TASK_PATH, "(%.1f MB)" % (os.path.getsize(TASK_PATH) / 1e6))
else:
    print("모델 번들 다운로드:", TASK_URL)
    try:
        urllib.request.urlretrieve(TASK_URL, TASK_PATH + ".part")
        os.replace(TASK_PATH + ".part", TASK_PATH)
        print("  저장:", TASK_PATH, "(%.1f MB)" % (os.path.getsize(TASK_PATH) / 1e6))
    except Exception as e:
        print("  [주의] 다운로드 실패:", e, "— 네트워크 확인 후 이 셀을 다시 ▶ 실행하세요.")

# --- 안전 점검: mediapipe 가 우리 핀을 덮어썼는가? ---
# (session2 에서 ultralytics 가 torch 를 덮어썼는지 검사한 것과 같은 방어 패턴)
print("\n[설치 후 버전 — 덮어쓰기 검사]")
changed = []
for mod in ["numpy", "torch", "ultralytics"]:
    try:
        # 이미 import 된 모듈은 캐시된 버전을 보여줄 수 있으므로 pip 이 아는 값을 함께 본다.
        now = getattr(__import__(mod), "__version__", "?")
    except Exception:
        now = "(없음)"
    mark = ""
    if now != BEFORE[mod]:
        changed.append("%s: %s -> %s" % (mod, BEFORE[mod], now))
        mark = "   <- 바뀜!"
    print("  -", mod, ":", now, mark)
try:
    import mediapipe as mp
    print("  - mediapipe :", mp.__version__, " (핀 없이 설치된 최신 버전)")
except Exception as e:
    print("  - mediapipe : import 실패 —", e)
    print("    런타임 > 세션 다시 시작 후 이 셀만 다시 ▶ 실행하면 대개 해결됩니다.")

if changed:
    print("\n[경고] mediapipe 설치로 버전이 바뀌었습니다:", changed)
    print("       -> 01_train_lab 을 다시 돌려야 한다면 런타임을 새로 시작하세요.")
    print("       -> 이 값들을 기록해 두면 나중에 mediapipe 핀을 확정할 때 근거가 됩니다.")
else:
    print("\n[확인] numpy/torch/ultralytics 버전이 그대로입니다 — 덮어쓰기 없음.")

In [ ]:
# 셀 3 · 사진 1장 확보 (웹캠 -> 업로드 -> 샘플 이미지, 3단 폴백)
# 수업 중 웹캠 권한 거부는 흔하다. 어떤 경우에도 이 셀은 예외로 죽지 않는다.
import os, urllib.request

IMG_PATH = os.path.join("data", "webcam.jpg")
SAMPLE_URL = "https://storage.googleapis.com/mediapipe-tasks/hand_landmarker/woman_hands.jpg"
SOURCE = None

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def try_webcam():
    """코랩에서 JS getUserMedia 로 정지 사진 1장. (라이브 영상 스트림은 범위 밖)"""
    if not IN_COLAB:
        print("  (로컬이라 웹캠 캡처를 건너뜁니다)")
        return False
    try:
        from IPython.display import display, Javascript
        from google.colab.output import eval_js
        from base64 import b64decode
        js = Javascript("""
        async function takePhoto(quality) {
          const div = document.createElement('div');
          const btn = document.createElement('button');
          btn.textContent = '📸 사진 찍기 (클릭)';
          btn.style.cssText = 'margin:6px;padding:8px 14px;font-size:15px;';
          const video = document.createElement('video');
          video.style.display = 'block';
          video.style.maxWidth = '480px';
          const stream = await navigator.mediaDevices.getUserMedia({video: true});
          document.body.appendChild(div);
          div.appendChild(btn);
          div.appendChild(video);
          video.srcObject = stream;
          await video.play();
          google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
          await new Promise((resolve) => btn.onclick = resolve);
          const canvas = document.createElement('canvas');
          canvas.width = video.videoWidth;
          canvas.height = video.videoHeight;
          canvas.getContext('2d').drawImage(video, 0, 0);
          stream.getVideoTracks()[0].stop();
          div.remove();
          return canvas.toDataURL('image/jpeg', quality);
        }
        """)
        display(js)
        print("  브라우저가 카메라 권한을 물으면 허용하고, 화면의 '사진 찍기' 버튼을 누르세요.")
        data = eval_js("takePhoto(0.92)")
        with open(IMG_PATH, "wb") as f:
            f.write(b64decode(data.split(",")[1]))
        return os.path.getsize(IMG_PATH) > 0
    except Exception as e:
        print("  웹캠 실패/거부:", type(e).__name__, e)
        return False


def try_upload():
    """파일 업로드 위젯 (코랩). 로컬이면 data/webcam.jpg 를 직접 두면 된다."""
    if not IN_COLAB:
        print("  (로컬) data/webcam.jpg 로 사진을 직접 넣어두면 그걸 씁니다.")
        return os.path.exists(IMG_PATH) and os.path.getsize(IMG_PATH) > 0
    try:
        from google.colab import files
        print("  사진 파일을 하나 올려 주세요(취소해도 됩니다 — 샘플 이미지로 넘어갑니다).")
        up = files.upload()
        if not up:
            return False
        name = list(up.keys())[0]
        with open(IMG_PATH, "wb") as f:
            f.write(up[name])
        return os.path.getsize(IMG_PATH) > 0
    except Exception as e:
        print("  업로드 실패/취소:", type(e).__name__, e)
        return False


def try_sample():
    try:
        print("  샘플 이미지 다운로드:", SAMPLE_URL)
        urllib.request.urlretrieve(SAMPLE_URL, IMG_PATH)
        return os.path.getsize(IMG_PATH) > 0
    except Exception as e:
        print("  샘플 다운로드 실패:", type(e).__name__, e)
        return False


print("1) 웹캠으로 사진 1장 시도")
if try_webcam():
    SOURCE = "webcam"
else:
    print("2) 파일 업로드로 폴백")
    if try_upload():
        SOURCE = "upload"
    else:
        print("3) 샘플 이미지로 폴백")
        if try_sample():
            SOURCE = "sample"

if SOURCE is None:
    print("\n[주의] 사진을 확보하지 못했습니다. 네트워크 확인 후 이 셀(셀 3)을 다시 ▶ 실행하세요.")
    print("       (셀은 죽지 않았습니다 — 셀 4·5 는 사진이 있어야 동작합니다.)")
else:
    from PIL import Image
    import matplotlib.pyplot as plt
    im = Image.open(IMG_PATH)
    print("\n확보:", IMG_PATH, "| 출처:", SOURCE, "| 크기: %dx%d" % im.size)
    fig, ax = plt.subplots(figsize=(6.0, 4.5))
    ax.imshow(im); ax.axis("off"); ax.set_title("input photo (source: %s)" % SOURCE)
    plt.tight_layout(); plt.show()

In [ ]:
# 셀 4 · 손 랜드마크 21개 (MediaPipe Tasks API)
# 레거시 mp.solutions.hands 는 2023년 폐기됐다 — 최신 버전에서 동작이 보장되지 않으므로 Tasks API 를 쓴다.
import os
import matplotlib.pyplot as plt
from PIL import Image

IMG_PATH = os.path.join("data", "webcam.jpg")
TASK_PATH = os.path.join("weights", "hand_landmarker.task")
if not os.path.exists(IMG_PATH):
    raise RuntimeError("사진이 없습니다: %s — 셀 3을 먼저 ▶ 실행하세요." % IMG_PATH)
if not os.path.exists(TASK_PATH):
    raise RuntimeError("모델 번들이 없습니다: %s — 셀 2를 먼저 ▶ 실행하세요." % TASK_PATH)

import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

base = mp_python.BaseOptions(model_asset_path=str(TASK_PATH))
opts = vision.HandLandmarkerOptions(base_options=base, num_hands=2)
with vision.HandLandmarker.create_from_options(opts) as lm:
    result = lm.detect(mp.Image.create_from_file(str(IMG_PATH)))

# 손 뼈대 연결(21점 고정 위상) — 레거시 모듈에 의존하지 않도록 직접 적는다.
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),            # thumb
    (0, 5), (5, 6), (6, 7), (7, 8),            # index
    (5, 9), (9, 10), (10, 11), (11, 12),       # middle
    (9, 13), (13, 14), (14, 15), (15, 16),     # ring
    (13, 17), (17, 18), (18, 19), (19, 20),    # pinky
    (0, 17),                                   # palm
]

im = Image.open(IMG_PATH).convert("RGB")
W, H = im.size
hands = list(getattr(result, "hand_landmarks", []) or [])

fig, ax = plt.subplots(figsize=(7.5, 5.6))
ax.imshow(im); ax.axis("off")
for hi, marks in enumerate(hands):
    xs = [m.x * W for m in marks]
    ys = [m.y * H for m in marks]
    for a, b in HAND_CONNECTIONS:
        ax.plot([xs[a], xs[b]], [ys[a], ys[b]], linewidth=2.0, color="lime", alpha=0.9)
    ax.scatter(xs, ys, s=22, c="red", zorder=3)

sides = []
for h in (getattr(result, "handedness", []) or []):
    try:
        sides.append(h[0].category_name)
    except Exception:
        sides.append("?")

if hands:
    ax.set_title("MediaPipe HandLandmarker — %d hand(s), 21 landmarks each  [%s]"
                 % (len(hands), ", ".join(sides) if sides else "-"))
else:
    ax.set_title("MediaPipe HandLandmarker — no hand detected")
plt.tight_layout()
plt.savefig(os.path.join("outputs", "04_hand.png"), dpi=130)
plt.show()
print("저장: outputs/04_hand.png")

print()
if not hands:
    # 예외로 죽이지 않고 친절히 알린다.
    print("손이 검출되지 않았습니다. 사진에 손이 크게, 밝게 나오도록 다시 찍어 보세요(셀 3 -> 셀 4).")
    print("샘플 이미지로 확인하려면 data/webcam.jpg 를 지우고 셀 3에서 웹캠/업로드를 건너뛰면 됩니다.")
else:
    print("검출된 손: %d개" % len(hands))
    for i, marks in enumerate(hands):
        print("  손 %d: 좌/우 = %s | 랜드마크 %d개" % (i + 1, sides[i] if i < len(sides) else "?", len(marks)))
    print("손 하나당 랜드마크는 항상 21개입니다 — 손가락 마디 위치가 좌표로 나옵니다.")

In [ ]:
# 셀 5 · 전신 포즈 17개 키포인트 (YOLO11n-pose)
# ultralytics 는 requirements.txt 에 이미 핀돼 있다 — 새 의존성 0개.
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO

IMG_PATH = os.path.join("data", "webcam.jpg")
if not os.path.exists(IMG_PATH):
    raise RuntimeError("사진이 없습니다: %s — 셀 3을 먼저 ▶ 실행하세요." % IMG_PATH)

model = YOLO("yolo11n-pose.pt")            # 3회차의 그 YOLO — 가중치만 pose 용
res = model(IMG_PATH, verbose=False)[0]

n_person = len(res.boxes) if res.boxes is not None else 0
n_kpt = 0
try:
    n_kpt = int(res.keypoints.data.shape[1]) if res.keypoints is not None and n_person else 0
except Exception:
    pass

fig, ax = plt.subplots(figsize=(7.5, 5.6))
ax.imshow(res.plot()[:, :, ::-1])          # ultralytics 가 17점 스켈레톤을 그려 준다 (BGR -> RGB)
ax.axis("off")
ax.set_title("YOLO11n-pose — %d person(s), %d COCO keypoints each" % (n_person, n_kpt))
plt.tight_layout()
plt.savefig(os.path.join("outputs", "04_pose.png"), dpi=130)
plt.show()
print("저장: outputs/04_pose.png")

print()
if n_person == 0:
    print("사람이 검출되지 않았습니다(손만 찍힌 사진이면 정상입니다). 전신이 나오게 다시 찍어 보세요.")
else:
    print("검출된 사람: %d명 | 사람당 키포인트: %d개" % (n_person, n_kpt))
print("3회차에서 본 그 YOLO 입니다. 백본은 그대로, 헤드만 바뀌어 '네모'가 '관절'이 됐습니다.")

In [ ]:
# 셀 6 · 비교 정리 (표만 출력 — 학습 없음)
import unicodedata


def pad(s, n):
    """한글은 터미널에서 두 칸을 차지하므로 글자 수가 아니라 '표시 폭'으로 맞춘다."""
    w = sum(2 if unicodedata.east_asian_width(ch) in "WF" else 1 for ch in s)
    return s + " " * max(0, n - w)


rows = [
    ("찾는 것",   "손",                 "사람"),
    ("키포인트",  "21",                 "17"),
    ("라이선스",  "Apache-2.0",         "AGPL-3.0"),
]
print("=" * 62)
print(pad("", 12) + pad("MediaPipe HandLandmarker", 26) + pad("YOLO11n-pose", 22))
print("-" * 62)
for k, a, b in rows:
    print(pad(k, 12) + pad(a, 26) + pad(b, 22))
print("=" * 62)
print()
print("라이선스: 3회차의 라이선스 함정이 여기서도 그대로 적용됩니다.")
print("  - MediaPipe(Apache-2.0): 비교적 자유롭게 쓰고 배포할 수 있습니다.")
print("  - YOLO11(AGPL-3.0): 이 모델을 쓴 서비스를 배포하면 '내 코드도 같은 라이선스로 공개' 의무가 생깁니다.")
print("    연구실 논문·내부 실험은 괜찮지만, 서비스로 내보낼 계획이면 상용 라이선스를 확인하세요.")
print()
print("같은 '키포인트 검출'이라도 무엇을 몇 개 찾는지, 어떤 조건으로 쓸 수 있는지가 다릅니다.")
print("모델을 고를 때는 성능만 보지 말고 라이선스도 같이 보세요.")